<a href="https://colab.research.google.com/github/MR-Goldenyt/Argus/blob/main/Argus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# IT7009 Artificial Intelligence - Selected Project Brief #2

## Intelligent Cyberattack Analysis System


## Phase 1: Data Cleaning and Preprocessing


### i. Imports and Data Loading

In [70]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Upload this file to the Colab session storage first
DATASET_PATH = "Dataset_Brief_2.csv"
df = pd.read_csv(DATASET_PATH, low_memory=False)

# Display the raw dataset
print("Raw dataset shape:", df.shape)
df.head()

Raw dataset shape: (48826, 47)


,flow_duration,Header_Length,Protocol Type,Duration,Rate,Srate,Drate,fin_flag_number,syn_flag_number,rst_flag_number,...,Std,Tot size,IAT,Number,Magnitue,Radius,Covariance,Variance,Weight,label
0,4.835233,1713517.54,16.89,66.33,634.919224,634.919224,0.0,0.0,0.0,0.0,...,23.51914405190254,540.23,8.375931e+07,9.5,32.983477,31.837886,3663.34722,0.14,141.55,Mirai-udpplain
1,0.000000,54.0,6.0,64.00,2.671835,2.671835,0.0,0.0,1.0,0.0,...,0.0,54.0,8.297303e+07,9.5,10.392305,0.000000,0.00000,0.0,141.55,DoS-SYN_Flood
2,0.000000,54.0,6.0,64.00,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.0,54.0,8.333120e+07,9.5,10.392305,0.000000,0.00000,0.0,141.55,DDoS-PSHACK_Flood
3,0.000000,0.0,1.0,64.00,160.127665,160.127665,0.0,0.0,0.0,0.0,...,0.0,42.0,8.312889e+07,9.5,9.165151,0.000000,0.00000,0.0,141.55,DDoS-ICMP_Flood
4,0.000000,54.0,6.0,64.00,1.262448,1.262448,0.0,1.0,0.0,1.0,...,0.0,54.0,8.334516e+07,9.5,10.392305,0.000000,0.00000,0.0,141.55,DDoS-RSTFINFlood


### ii. Data Analysis of Raw Data

In [71]:
# Display Data Analysis

# Brief Information
df.info()

# Statistical Summary
df.describe(include='all').T.head(20)

<class 'pandas.DataFrame'>
RangeIndex: 48826 entries, 0 to 48825
Data columns (total 47 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   flow_duration    48815 non-null  float64
 1   Header_Length    48815 non-null  str    
 2   Protocol Type    48816 non-null  str    
 3   Duration         48816 non-null  float64
 4   Rate             48814 non-null  float64
 5   Srate            48816 non-null  float64
 6   Drate            48811 non-null  float64
 7   fin_flag_number  48814 non-null  float64
 8   syn_flag_number  48816 non-null  float64
 9   rst_flag_number  48815 non-null  float64
 10  psh_flag_number  48815 non-null  str    
 11  ack_flag_number  48814 non-null  float64
 12  ece_flag_number  48816 non-null  float64
 13  cwr_flag_number  48816 non-null  float64
 14  ack_count        48816 non-null  float64
 15  syn_count        48813 non-null  float64
 16  fin_count        48815 non-null  float64
 17  urg_count        48816 

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
flow_duration,48815.0,NaN,NaN,NaN,14.495303,382.599894,-999.0,0.0,0.0,0.128624,47629.094561
Header_Length,48815,15623,54.0,14557,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Protocol Type,48816,915,6.0,22721,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Duration,48816.0,NaN,NaN,NaN,66.276401,12.9069,0.0,64.0,64.0,64.0,255.0
Rate,48814.0,NaN,NaN,NaN,8664.10689,89626.647942,-999.0,2.095422,15.69341,128.325003,3145728.0
Srate,48816.0,NaN,NaN,NaN,8663.824629,89624.823444,-999.0,2.095305,15.69341,128.351765,3145728.0
Drate,48811.0,NaN,NaN,NaN,0.000001,0.000089,0.0,0.0,0.0,0.0,0.014636
fin_flag_number,48814.0,NaN,NaN,NaN,0.083583,0.276764,0.0,0.0,0.0,0.0,1.0
syn_flag_number,48816.0,NaN,NaN,NaN,0.204359,0.403237,0.0,0.0,0.0,0.0,1.0
rst_flag_number,48815.0,NaN,NaN,NaN,0.088559,0.284109,0.0,0.0,0.0,0.0,1.0


### iii. Categorical, Temporal, and Identifier Audit

In [72]:
feature_cols = [c for c in df.columns if c != 'label']
print("Number of feature columns:", len(feature_cols))
print(f"\nColumn dtypes as read:\n{df.dtypes.value_counts()}")
print("\nAny column name suggesting an identifier/date field?")
suspect = [c for c in df.columns if any(k in c.lower() for k in ['id', 'date', 'time', 'timestamp']) and c != 'flow_duration']
print(suspect if suspect else "None found (besides the numeric duration/IAT features, which are not timestamps).")
print("\nTarget classes:", df['label'].nunique(), "| Missing labels:", df['label'].isnull().sum())

Number of feature columns: 46

Column dtypes as read:
float64    34
str        13
Name: count, dtype: int64

Any column name suggesting an identifier/date field?
None found (besides the numeric duration/IAT features, which are not timestamps).

Target classes: 33 | Missing labels: 0


### iv. Inconsistent Values and Incorrect Data Types

`df.info()` above shows several columns that should be purely numeric (e.g. `Header_Length`,
`Protocol Type`, etc.) are instead read as `object` type. Inspecting the non-numeric
entries reveals the literal text tokens **`"invalid"`** and **`"unknown"`** planted inside otherwise
numeric cells not a genuine category.

**Proposed Solution:** replace these text tokens with `NaN`, then impute every feature column to numeric dtype.


In [73]:
# Features only dataframe
features = df[feature_cols]

# Find non-numeric values
is_non_numeric = features.map(
    lambda x: pd.notna(x) and pd.isna(pd.to_numeric(x, errors='coerce'))
)

# Get the unique non-numeric values
unique_non_numbers = features.stack()[is_non_numeric.stack()].unique()

# Print unique non-numeric values
print("Identfied non-numeric values:", unique_non_numbers)

# Print Feature cols with non-numeric values
object_cols_before = df[feature_cols].select_dtypes(include=["str","object"]).columns.tolist() # note: use "str" in include to avoid pandas warning
print("\nFeature columns containing non-numeric values incorrectly typed as text before fixing:\n", object_cols_before,sep="")

# Clean inconsistent values by replacing and coercing as nan
df[feature_cols] = df[feature_cols].replace(unique_non_numbers, np.nan)
for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Print feature cols that are still non-numeric (should be empty)
object_cols_after = df[feature_cols].select_dtypes(include=["str","object"]).columns.tolist()
print("\nFeature columns still non-numeric after fixing (should be empty):", object_cols_after)
print("All 46 feature columns are now numeric:", len(object_cols_after) == 0)


Identfied non-numeric values: ['unknown' 'invalid']

Feature columns containing non-numeric values incorrectly typed as text before fixing:
['Header_Length', 'Protocol Type', 'psh_flag_number', 'rst_count', 'HTTPS', 'ICMP', 'IPv', 'LLC', 'Min', 'Std', 'Tot size', 'Variance']

Feature columns still non-numeric after fixing (should be empty): []
All 46 feature columns are now numeric: True
